# DumpWatch 2.0 reproduction
Run the frozen models only. Archived prediction checks are separate from fresh inference. See the package README.

## 1. Setup
Mount Drive read-only. Use a GPU if available for Task A.

In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get("DUMPWATCH_PACKAGE", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "project_setup.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within the unpacked code folder or set DUMPWATCH_PACKAGE to it. See SETUP.md.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from project_setup import project_paths, new_output, enable_demo_vendor
P = project_paths()
enable_demo_vendor()
REPRO = ROOT / "Reproduction"
sys.path[:0] = [str(REPRO / "Task_A"), str(REPRO)]
from reproduce import task_a, task_b
RESULTS = new_output("frozen_evaluation")
print("Task A dataset:", P["task_a_root"])
print("Task B archive:", P["real_zip"])
print("Inference only; new outputs:", RESULTS)


## 2. Task A · Video holdout
Checks the original 100-clip partition, video hashes and timing policy. Fresh inference requires the authorized videos.

In [ ]:
a = task_a(REPRO / "Task_A", P["task_a_root"], RESULTS)


## 3. Task B · Cleanup holdout
Uses the nine locked pairs and original scoring cells.

In [ ]:
b = task_b(REPRO / "Task_B", P["real_zip"], RESULTS)


## 4. Results verification
The runners compare fresh metrics against the archives and raise on a mismatch.

In [ ]:
if a is None:
    print("Task A: archived prediction check only. No fresh evaluation.")
else:
    print(f"Task A fresh: accuracy {a['accuracy']:.3f}, macro-F1 {a['macro_f1']:.3f}, event F1 {a['event_f1']:.3f}, "
          f"onset MAE {a['onset_mae_s']:.2f} s with {a['timing_reported']}/{a['timing_eligible']} coverage.")
if b is None:
    print("Task B: archived prediction check only. No fresh evaluation.")
else:
    print(f"Task B fresh: {b['forced']['correct_accepted']}/{b['forced']['n']} forced correct, "
          f"{b['fixed']['accepted']} accepted, {b['fixed']['abstained']} abstentions, "
          f"{b['fixed']['false_clearance_count']} accepted false clearances.")
if a is None or b is None:
    print("FULL REPRODUCTION INCOMPLETE. Supply authorized missing evaluation inputs.")
print("New files are stored only in:", RESULTS)